# Лабораторная работа 4

Цель работы:  
Сформировать начальный практический навык оптимизации гиперпараметров модели машинного обучения с использованием библиотек HyperOpt и Optuna на табличных данных.

Инструкция:
- Запускайте ячейки по очереди, если не указано иное действие.
- В конце выполните самостоятельное задание.

При первом запуске установите необходимые библиотеки.  
Раскомментируйте строку и запустите ячейку

In [ ]:
# !uv add hyperopt optuna

Используется результат лабораторной работы 2 в виде файла с БД
- labs/lab_02/data/db/data_lab02_prepared.db'

# 1. Подготовка данных и обучение базовой модели

Задача: Отличить внутрироссийские перевозки от экспортных.

- Класс 0: ВНУТРИРОССИЙСКАЯ
- Класс 1: ЭКСПОРТ

Остальные типы признака не учитывать.

## 1.1. Загрузка данных

In [ ]:
import os
import joblib
import optuna
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from datetime import datetime
from sqlalchemy import create_engine
from hyperopt import hp, fmin, tpe, Trials, STATUS_OK
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import roc_auc_score, roc_curve, auc

In [ ]:
sns.set()
optuna.logging.set_verbosity(optuna.logging.WARNING)
RANDOM_STATE = 42
N_SPLITS = 3
DB_NAME = 'data_lab02_prepared'
DB_PATH = f'../lab_02/data/db/{DB_NAME}.db'

In [ ]:
# Загрузка таблицы
engine = create_engine(f'sqlite:///{DB_PATH}')

query_advanced = f"""
    SELECT *
    FROM {DB_NAME}
"""
df = pd.read_sql_query(query_advanced, engine)

## 1.2. Разведочный анализ (EDA)

In [ ]:
df.head()

In [ ]:
df.info()

## 1.3. Формирование целевой переменной

In [ ]:
target_col = "Режим перевозки_расч."

In [ ]:
# 1. Оставляем только два нужных класса
classes_to_keep = ["ЭКСПОРТ", "ВНУТРИРОССИЙСКАЯ"]
df = df[df[target_col].isin(classes_to_keep)].copy()
print(f"{df.shape=}")

In [ ]:
# 2. Формируем бинарный таргет
# 1 для ЭКСПОРТА, 0 для ВНУТРИРОССИЙСКОЙ
df["target"] = (df[target_col] == "ЭКСПОРТ").astype(int)

In [ ]:
# 3. Проверка баланса классов (студент должен увидеть примерно 50/50)
print("Распределение классов после фильтрации:")
print(df["target"].value_counts())
print("\nДоли классов:")
print(df["target"].value_counts(normalize=True))

## 1.4. Создание новых признаков

In [ ]:
df["Дата операции"] = pd.to_datetime(df["Дата операции"], errors="coerce")
df["Дата время операции отправки"] = pd.to_datetime(
    df["Дата время операции отправки"],
    errors="coerce"
)

df["month"] = df["Дата операции"].dt.month
df["dayofweek"] = df["Дата операции"].dt.dayofweek
df["hour"] = df["Дата время операции отправки"].dt.hour

## 1.5. Выбор признаков

In [ ]:
# Категориальные признаки
cat_cols = [
    "Классификатор перевозок",
    "Тип заказа",
    "Ранг отправки",
    "Операция",
    "Тип услуги",
    "Наименование плановой услуги предоставления",
    "Связка",
    "Тип клиента"
]
# Числовые признаки
num_cols = [
    "ДФЭ",
    "month",
    "dayofweek",
    "hour",
]
# Целевой признак
target_col = "target"

feature_cols = cat_cols + num_cols

In [ ]:
print("Количество уникальных значений в категориальных признаках:")
print(df[cat_cols].nunique().sort_values(ascending=False))

## 1.6. Разделение на обучающую и тестовую выборки

In [ ]:
X = df[feature_cols].copy()
y = df[target_col]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    stratify=y,
    random_state=42
)

## 1.7. Предобработка категориальных признаков

Полезно посмотреть, сколько признаков было до one-hot кодирования

In [ ]:
print("Размер X_train до one-hot:", X_train.shape)
print("Размер X_test до one-hot:", X_test.shape)

Кодируем только категориальные колонки

In [ ]:
X_train = pd.get_dummies(X_train, columns=cat_cols, dtype=np.uint8)
X_test = pd.get_dummies(X_test, columns=cat_cols, dtype=np.uint8)

После этого категориальные колонки исчезнут, а вместо них появятся новые бинарные признаки вида:
- Клиент_Тип 1
- Клиент_Тип 2
- Тип заказа_Тип 1
- Тип заказа_Тип 2
- ...

При этом в train и test могут получиться разные наборы колонок. 
Например, какая-то категория есть только в test или только в train.

Выровняем состав признаков:
- оставляем только те колонки, которые есть в X_train;
- если в X_test есть колонка, которой не было в X_train, она удаляется;
- если в X_test нет какой-то колонки из X_train, она создаётся и заполняется нулями.

In [ ]:
X_train, X_test = X_train.align(
    X_test,
    join="left",
    axis=1,
    fill_value=0
)

In [ ]:
print("Размер X_train после one-hot:", X_train.shape)
print("Размер X_test после one-hot:", X_test.shape)

## 1.8. Подготовка числовых признаков

В рамках лабораторной работы данный этап пропущен, чтобы не перегружать объем.

## 1.9. Обучение базовой модели

Значения параметров модели были выбраны вручную «на глаз».

Но как наилучшие?

Можно перебрать все возможные комбинации вручную (Grid Search), но это занимает очень много времени, особенно если параметров много и каждый из них может принимать множество значений.

Optuna и Hyperopt решают эту проблему:
- Автоматизация: Они сами перебирают различные комбинации гиперпараметров.
- Эффективность: Вместо полного перебора всех вариантов (как Grid Search), они используют умные алгоритмы поиска.
- Масштабируемость: Они поддерживают параллельные вычисления, что позволяет ускорить процесс, используя несколько ядер процессора или даже несколько машин

In [ ]:
# Алгоритм машинного обучения, который можно представить как «совет экспертов», где эксперт - одно решающее дерево.
baseline_model = RandomForestClassifier(
    n_estimators=11, 
    max_depth=3,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

# Модель обучается на n_splits-1 частях (фолдах), проверяется на оставшейся. Так мы получаем усреднённую оценку качества, 
# более надёжную, чем по одному случайному сплиту.
cv = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE
)

# Оценки по разбиениям и усреднённая оценка
baseline_cv_scores = cross_val_score(
    baseline_model,
    X_train,
    y_train,
    cv=cv,
    scoring="roc_auc",
    n_jobs=1
)
baseline_cv = baseline_cv_scores.mean()
print(f"Оценки на фолдах: {baseline_cv_scores}")

# После кросс-валидации модель переобучается на всех данных X_train, y_train. 
# Это делается для того, чтобы получить финальную модель, готовую к использованию (или для сравнения с тестом)
baseline_model.fit(X_train, y_train)

# Метрика качества ROC AUC — площадь под ROC-кривой 
# (подходит для бинарной классификации, показывает способность модели разделять классы)
baseline_test = roc_auc_score(
    y_test,
    baseline_model.predict_proba(X_test)[:, 1]
)

print("Baseline CV ROC-AUC:", baseline_cv)
print("Baseline Test ROC-AUC:", baseline_test)

Сохранение модели

In [ ]:
# Создаём папку для результатов
timestamp = datetime.now().strftime("%Y%m%d_%H-%M-%S")
output_dir = f"results/{timestamp}"
os.makedirs(output_dir, exist_ok=True)

In [ ]:
# Сохраняем модель
model_path = os.path.join(output_dir, f"baseline_model.joblib")
joblib.dump(baseline_model, model_path)
print(f"Модель сохранена в файл: {model_path}")

Оценки:
- CV ROC-AUC – усреднённая оценка по фолдам (характеризует стабильность модели на разных подвыборках).
- Test ROC-AUC – оценка на отложенной выборке (имитация работы на новых данных).

Если сравнить эти числа:
- CV ≈ Test – модель хорошо обобщается, переобучения нет.
- CV >> Test – скорее всего, модель переобучилась или есть расхождение в распределении train/test.
- CV << Test – редкий случай, обычно говорит о том, что тестовая выборка слишком простая или мала.

# 2. Подбор гиперпараметров

Ниже представлены две библиотеки для подбора гиперпараметров - HyperOpt и Optune

## 2.1. HyperOpt

Функция создания модели

In [ ]:
def make_rf_model(params):
    return RandomForestClassifier(
        n_estimators=int(params.get("n_estimators", 200)),
        max_depth=int(params.get("max_depth", 15)),
        min_samples_split=int(params.get("min_samples_split", 2)),
        min_samples_leaf=int(params.get("min_samples_leaf", 1)),
        max_features=params.get("max_features", "sqrt"),
        random_state=RANDOM_STATE,
        n_jobs=-1
    )

Функция оценки

In [ ]:
def evaluate_params(params):
    model = make_rf_model(params)

    scores = cross_val_score(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring="roc_auc",
        n_jobs=1
    )
    print(f"Оценки на фолдах: {scores.tolist()}")
    return float(scores.mean())

Пространство поиска

In [ ]:
space = {
    "n_estimators": hp.quniform("n_estimators", 100, 300, 50),
    "max_depth": hp.quniform("max_depth", 5, 20, 1),
    "min_samples_split": hp.quniform("min_samples_split", 2, 20, 1),
    "min_samples_leaf": hp.quniform("min_samples_leaf", 1, 10, 1),
    "max_features": hp.uniform("max_features", 0.3, 0.7)
}

В строке 
```python
hp.quniform("n_estimators", 100, 500, 50))
```
- "n_estimators" - название параметра
- 100 - левая граница поиска
- 500 - права граница поиска
- 50 - шаг


Целевая функция

In [ ]:
def hyperopt_objective(params):
    score = evaluate_params(params)

    return {
        "loss": 1.0 - score,
        "status": STATUS_OK
    }

Запуск

In [ ]:
trials = Trials()

best_hyperopt_params = fmin(
    fn=hyperopt_objective,
    space=space,
    algo=tpe.suggest,
    max_evals=3,
    trials=trials,
    rstate=np.random.default_rng(RANDOM_STATE)
)

best_hyperopt_params = {
    "n_estimators": int(best_hyperopt_params["n_estimators"]),
    "max_depth": int(best_hyperopt_params["max_depth"]),
    "min_samples_split": int(best_hyperopt_params["min_samples_split"]),
    "min_samples_leaf": int(best_hyperopt_params["min_samples_leaf"]),
    "max_features": best_hyperopt_params["max_features"]
}

print("Лучшие параметры HyperOpt:", best_hyperopt_params)

Оценка

In [ ]:
hyperopt_cv_score = evaluate_params(best_hyperopt_params)

hyperopt_model = make_rf_model(best_hyperopt_params)
hyperopt_model.fit(X_train, y_train)

hyperopt_test_score = roc_auc_score(
    y_test,
    hyperopt_model.predict_proba(X_test)[:, 1]
)

print("HyperOpt CV ROC-AUC:", hyperopt_cv_score)
print("HyperOpt Test ROC-AUC:", hyperopt_test_score)

Сохраняем модель

In [ ]:
# Сохраняем модель
model_path = os.path.join(output_dir, f"hyperopt_model.joblib")
joblib.dump(hyperopt_model, model_path)
print(f"Модель сохранена в файл: {model_path}")

## 2.2. Optuna 

Функция создания модели

In [ ]:
def optuna_objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 300, step=50),
        "max_depth": trial.suggest_int("max_depth", 5, 20),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 20),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 10),
        "max_features": trial.suggest_float("max_features", 0.3, 0.7),
    }
    return evaluate_params(params)

Запуск

In [ ]:
study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE)
)

study.optimize(
    optuna_objective,
    n_trials=3,
    timeout=600
)

best_optuna_params = study.best_params

print("Лучшие параметры Optuna:", best_optuna_params)
print("Лучший CV ROC-AUC:", study.best_value)

Оценка

In [ ]:
optuna_cv_score = study.best_value

optuna_model = make_rf_model(best_optuna_params)
optuna_model.fit(X_train, y_train)

optuna_test_score = roc_auc_score(
    y_test,
    optuna_model.predict_proba(X_test)[:, 1]
)

print("Optuna CV ROC-AUC:", optuna_cv_score)
print("Optuna Test ROC-AUC:", optuna_test_score)

Сохранение модели

In [ ]:
# Сохраняем модель
model_path = os.path.join(output_dir, f"optuna_model.joblib")
joblib.dump(optuna_model, model_path)
print(f"Модель сохранена в файл: {model_path}")

## 2.3. Итоговая таблица результатов

In [ ]:
results = pd.DataFrame({
    "Метод": [
        "Baseline",
        "HyperOpt",
        "Optuna"
    ],
    "CV ROC-AUC": [
        baseline_cv,
        hyperopt_cv_score,
        optuna_cv_score
    ],
    "Test ROC-AUC": [
        baseline_test,
        hyperopt_test_score,
        optuna_test_score
    ]
})

print(results)

Сохраняем метрики

In [ ]:
metrics_path = os.path.join(output_dir, f"metrics.csv")
results.to_csv(metrics_path)

Визуализация

In [ ]:
# Получаем вероятности для всех моделей
y_pred_baseline = baseline_model.predict_proba(X_test)[:, 1]
y_pred_hyperopt = hyperopt_model.predict_proba(X_test)[:, 1]
y_pred_optuna = optuna_model.predict_proba(X_test)[:, 1]

# Вычисляем ROC-кривые
fpr_baseline, tpr_baseline, _ = roc_curve(y_test, y_pred_baseline)
fpr_hyperopt, tpr_hyperopt, _ = roc_curve(y_test, y_pred_hyperopt)
fpr_optuna, tpr_optuna, _ = roc_curve(y_test, y_pred_optuna)

# Вычисляем AUC
auc_baseline = auc(fpr_baseline, tpr_baseline)
auc_hyperopt = auc(fpr_hyperopt, tpr_hyperopt)
auc_optuna = auc(fpr_optuna, tpr_optuna)

# Строим график
plt.figure(figsize=(8, 6))

plt.plot(fpr_baseline, tpr_baseline, label=f'Baseline (AUC = {auc_baseline:.4f})', )
plt.plot(fpr_hyperopt, tpr_hyperopt, label=f'HyperOpt (AUC = {auc_hyperopt:.4f})', )
plt.plot(fpr_optuna, tpr_optuna, label=f'Optuna (AUC = {auc_optuna:.4f})')

# Диагональная линия (случайный классификатор)
plt.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Random Classifier (AUC = 0.5)')

plt.xlabel('False Positive Rate (FPR)')
plt.ylabel('True Positive Rate (TPR)')
plt.title('ROC Curves Comparison')
plt.legend(loc='lower right')
plt.grid(alpha=0.3)
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])

# Сохраняем график
plot_path = os.path.join(output_dir, f"ROC AUC.png")
plt.savefig(plot_path, dpi=300, bbox_inches="tight")
print(f"График сохранён: {plot_path}")

plt.tight_layout()
plt.show()

# 3. Самостоятельная работа

Задание:
1. Запустите ячейки и сохраните результат.
2. Повторите запуск, но в базовой модели удалите аргументы n_estimators, max_depth. Полученный код
`baseline_model = RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1)`
3. Повторите запуск, но добавьте в список числовых признаков "Сумма в RUB"
    - Получите результаты, сравните их с предыдущими. Обоснуйте изменения.

Вопросы (ответ в письменном виде):
- Для заданий 2-3 попробуйте объяснить, чем обусловлены наблюдаемые изменения.
- Какие основные этапы подготовки признаков пропущены и почему?
- Чем можно заполнять пустые значения (пропуски в данных) для категориальных признаков? Для числовых?


Подсказка:
- https://deepmachinelearning.ru/docs/Machine-learning/Data-preprocessing
- https://education.yandex.ru/handbook/ml/article/linear-models
- https://www.dmitrymakarov.ru/data/missing/